# BerlinMod Brussels - Vehicle-Commune Detection (JMEOS)

This notebook adapts **Exercise 4** of the AIS notebook to a Brussels scenario:
instead of detecting whether a ship is near a port, we detect which **Brussels commune**
each vehicle is in at each recorded instant.

**Key difference from Exercise 4:** both datasets are in SRID 3857 (Web Mercator, metres),
so no coordinate conversion is needed. This also lets us use the cleaner MEOS API path:
`geom_from_hexewkb` + `geo_to_stbox` instead of manual WKB parsing.

**Sections:**
- [1 - Setup (kernel, JARs, MEOS)](#n01)
- [2 - Load Brussels communes](#n02)
- [3 - Load BerlinMod vehicle positions](#n03)
- [4 - Approach 1: Brute Force (STBox first pass)](#n04)
- [5 - Approach 2: RTree index (STBox first pass)](#n05)
- [6 - Exact verification - eliminating false positives](#n06)
- [7 - Summary table (19 communes)](#n07)
- [8 - Scalability benchmark: 19 / 100 / 500 regions](#n08)


---
## <a id="n01"></a> 1 - Setup

### 1.1 Load JARs

In [1]:
%jars ../../../../jar/JMEOS-fat.jar

Add /workspace/src/main/java/examples/../../../../jar/JMEOS-fat.jar to classpath

### 1.2 Imports

In [2]:
import jnr.ffi.Pointer;
import jnr.ffi.LibraryLoader;
import java.util.*;
import java.io.*;
import java.util.stream.*;
import types.enums.RTreeSearchOp;
System.out.println("Imports OK");

Imports OK


### 1.3 MEOS interface

In [3]:
interface error_handler_fn {
    @jnr.ffi.annotations.Delegate
    void call(int level, int code, String message);
}

interface MeosBrussels {

    // Lifecycle
    void meos_initialize_timezone(String tz);
    void meos_initialize_error_handler(error_handler_fn handler);
    void meos_finalize();

    // Geometry input
    Pointer geo_from_text(String wkt, int srid);
    Pointer geom_from_hexewkb(String hex);

    // STBox
    Pointer stbox_in(String str);
    Pointer stbox_out(Pointer box, int maxdd);
    Pointer geo_to_stbox(Pointer geom);
    boolean overlaps_stbox_stbox(Pointer a, Pointer b);

    // STBox coordinate accessors (return double - no locale issue)
    // These bypass string serialization entirely and are a safe way
    // to retrieve bounding box coordinates on any locale.
    double stbox_xmin(Pointer box);
    double stbox_ymin(Pointer box);
    double stbox_xmax(Pointer box);
    double stbox_ymax(Pointer box);

    // Exact spatial check
    boolean geom_intersects2d(Pointer gs1, Pointer gs2);

    // RTree spatial index 
    Pointer rtree_create_stbox();
    void    rtree_insert(Pointer rtree, Pointer box, int id);
    Pointer rtree_search(Pointer rtree, int op, Pointer box, Pointer countPtr);
    void    rtree_free(Pointer rtree);
}

String ptrToString(Pointer p) {
    if (p == null) return "(null)";
    return p.getString(0);
}

System.out.println("Interface defined");

Interface defined


### 1.4 Load MEOS library and initialise

In [4]:
String MEOS_LIB_PATH = "/workspace/src";

MeosBrussels meos = LibraryLoader.create(MeosBrussels.class)
    .search(MEOS_LIB_PATH)
    .load("meos");

error_handler_fn errHandler = (lvl, code, msg) ->
    System.err.printf("[MEOS ERROR %d] %s%n", code, msg);

meos.meos_initialize_timezone("UTC");
meos.meos_initialize_error_handler(errHandler);
System.out.println("MEOS initialised");

MEOS initialised


---
## <a id="n02"></a> 2 - Load Brussels communes

The commune CSV contains 19 Brussels communes. Each geometry is stored as an EWKB hex
string in **SRID 3857** (Web Mercator, metres).

Instead of manually decoding the binary format (as done in Exercise 4 for UTM ports),
we call `geom_from_hexewkb` directly - MEOS handles all the binary parsing.
We then call `geo_to_stbox` to obtain the bounding box used by the RTree.
The four bbox coordinates are retrieved via `stbox_xmin/ymin/xmax/ymax` - direct `double`
accessors that bypass string serialisation and avoid any locale decimal-separator issue.

### 2.1 Commune record and CSV path

In [5]:
// Record holding everything we need per commune.
record CommuneBox(
    int id,
    String name,
    int population,
    Pointer geom,    // polygon geometry (SRID 3857)
    Pointer bbox,    // bounding STBOX - used for first-pass filtering
    double xmin, double ymin, // bbox corners in SRID 3857 metres
    double xmax, double ymax  // stored as doubles for easy offset arithmetic
) {}

// ↓ Adjust this path to where you placed brussels_communes.csv
String COMMUNES_FILE = "/workspace/src/main/java/examples/data/brussels_communes.csv";

System.out.println("Record and path defined");

Record and path defined


### 2.2 Parse and load communes

In [6]:
List<CommuneBox> communes = new ArrayList<>();

try (var reader = new BufferedReader(new FileReader(COMMUNES_FILE))) {
    reader.readLine(); // skip header: id,name,population,geom
    String line;
    while ((line = reader.readLine()) != null) {
        String[] f = line.split(",", 4);
        if (f.length < 4) continue;
        try {
            int    id   = Integer.parseInt(f[0].trim());
            String name = f[1].trim();
            int    pop  = Integer.parseInt(f[2].trim());
            String hex  = f[3].trim();

            Pointer geom = meos.geom_from_hexewkb(hex);
            if (geom == null) continue;

            Pointer bbox = meos.geo_to_stbox(geom);
            if (bbox == null) continue;

            // Use direct double accessors - locale-independent, no string parsing.
            double xmin = meos.stbox_xmin(bbox);
            double ymin = meos.stbox_ymin(bbox);
            double xmax = meos.stbox_xmax(bbox);
            double ymax = meos.stbox_ymax(bbox);

            communes.add(new CommuneBox(id, name, pop, geom, bbox,
                xmin, ymin, xmax, ymax));
        } catch (Exception e) {
            System.err.println("Skipping malformed line: " + e.getMessage());
        }
    }
}

System.out.printf("%d communes loaded%n", communes.size());
communes.forEach(c ->
    System.out.printf("  [%2d] %-40s pop=%,7d  bbox=(%.0f,%.0f)-(%.0f,%.0f)%n",
        c.id(), c.name(), c.population(),
        c.xmin(), c.ymin(), c.xmax(), c.ymax()));

19 communes loaded
  [ 1] Anderlecht                               pop=118,241  bbox=(472414,6587236)-(483153,6594900)
  [ 2] Auderghem - Oudergem                     pop= 33,313  bbox=(489353,6584186)-(498964,6590748)
  [ 3] Berchem-Sainte-Agathe - Sint-Agatha-Berchem pop= 24,701  bbox=(476358,6595733)-(480236,6599193)
  [ 4] Etterbeek                                pop=176,545  bbox=(487491,6589961)-(491116,6593862)
  [ 5] Evere                                    pop= 47,414  bbox=(488439,6595220)-(492772,6601142)
  [ 6] Forest - Vorst                           pop= 40,394  bbox=(478899,6585571)-(484592,6591427)
  [ 7] Ganshoren                                pop= 55,746  bbox=(477716,6597580)-(481580,6600466)
  [ 8] Ixelles - Elsene                         pop= 24,596  bbox=(484010,6586156)-(490185,6593151)
  [ 9] Jette                                    pop= 86,244  bbox=(477979,6597361)-(483201,6602592)
  [10] Koekelberg                               pop= 51,933  bbox=(479442,6595

---
## <a id="n03"></a> 3 - Load BerlinMod vehicle positions

The `berlinmod_instants.csv` file has columns: `tripid, vehid, day, seqno, geom, t`.

The `geom` column contains WKT points in SRID 3857 (`SRID=3857;POINT(x y)`) in metres.
Since the communes are also in SRID 3857, **no projection conversion is needed**.

For each record we create:
- a **STBOX** that collapses to a point - used as query key
- a **POINT geometry** used in the exact `geom_intersects2d` check

### 3.1 Vehicle position record and CSV path

In [7]:
record VehiclePos(
    int tripId,
    int vehId,
    String t,       // timestamp string (for display)
    double x,       // easting  in metres (SRID 3857)
    double y,       // northing in metres (SRID 3857)
    Pointer bbox,   // point STBOX - used as RTree / brute-force query
    Pointer geom    // POINT geometry - used in exact intersection check
) {}

// ↓ Adjust this path to where you placed berlinmod_instants.csv
String VEHICLES_FILE = "/workspace/src/main/java/examples/data/berlinmod_instants.csv";

System.out.println("Record and path defined");

Record and path defined


### 3.2 Parse and load vehicle positions

In [8]:
List<VehiclePos> positions = new ArrayList<>();
int skipped = 0;

try (var reader = new BufferedReader(new FileReader(VEHICLES_FILE))) {
    reader.readLine(); // skip header: tripid,vehid,day,seqno,geom,t
    String line;
    while ((line = reader.readLine()) != null) {
        // Split into at most 6 tokens.
        String[] f = line.split(",", 6);
        if (f.length < 6) continue;
        try {
            int    tripId = Integer.parseInt(f[0].trim());
            int    vehId  = Integer.parseInt(f[1].trim());
            // f[2] = day, f[3] = seqno - not needed here
            String geomStr = f[4].trim(); // e.g. "SRID=3857;POINT(482252.01 6580521.34)"
            String t       = f[5].trim();

            // Extract x, y from the WKT point (strip the SRID prefix first).
            String wkt = geomStr.contains(";") ? geomStr.split(";", 2)[1] : geomStr;
            // "POINT(x y)" → take the part between the parentheses
            String inner = wkt.substring(wkt.indexOf("(") + 1, wkt.indexOf(")")).trim();
            String[] coords = inner.split("\\s+");
            double x = Double.parseDouble(coords[0]);
            double y = Double.parseDouble(coords[1]);

            // A point STBOX: both corners collapse to the same coordinate.
            String boxStr = String.format(java.util.Locale.US,
                "SRID=3857;STBOX X((%f %f),(%f %f))", x, y, x, y);
            Pointer bbox = meos.stbox_in(boxStr);

            // Exact geometry used in geom_intersects2d.
            String pointWkt = String.format(java.util.Locale.US,
                "POINT(%f %f)", x, y);
            Pointer geom = meos.geo_from_text(pointWkt, 3857);

            if (bbox != null && geom != null)
                positions.add(new VehiclePos(tripId, vehId, t, x, y, bbox, geom));
            else skipped++;
        } catch (Exception e) { skipped++; }
    }
}

System.out.printf("%,d vehicle positions loaded, %d skipped%n",
    positions.size(), skipped);

// Quick preview
System.out.println();
System.out.printf("%-8s %-8s %-35s %s%n", "TripID", "VehID", "Timestamp", "Position (m)");
System.out.println("-".repeat(75));
positions.stream().limit(5).forEach(p ->
    System.out.printf("%-8d %-8d %-35s x=%.1f y=%.1f%n",
        p.tripId(), p.vehId(), p.t(), p.x(), p.y()));

216,075 vehicle positions loaded, 0 skipped

TripID   VehID    Timestamp                           Position (m)
---------------------------------------------------------------------------
2        2        2020-06-01 08:17:54.798+02          x=482252.0 y=6580521.3
2        2        2020-06-01 08:17:56.298+02          x=482251.9 y=6580526.3
2        2        2020-06-01 08:17:57.048+02          x=482251.8 y=6580531.3
2        2        2020-06-01 08:18:01.248+02          x=482250.8 y=6580566.3
2        2        2020-06-01 08:18:01.256062+02       x=482250.8 y=6580566.3


---
## <a id="n04"></a> 4 - Approach 1: Brute Force (STBox first pass)

For every vehicle position we loop over all 19 communes and call `overlaps_stbox_stbox`.
The bounding box of a commune is a rectangle that **circumscribes** the polygon:
a point in a corner of that rectangle may fall outside the actual commune boundary.
These are **false positives** - they are filtered out in Section 6.

**Complexity:** O(positions × communes) - acceptable for 19 communes,
but grows linearly if the number of regions increases.

In [9]:
record Match(int vehId, int tripId, String t, String commune) {}

List<Match> bruteResults = new ArrayList<>();
long t0brute = System.currentTimeMillis();

for (VehiclePos pos : positions) {
    for (CommuneBox com : communes) {
        if (meos.overlaps_stbox_stbox(pos.bbox(), com.bbox())) {
            bruteResults.add(new Match(pos.vehId(), pos.tripId(), pos.t(), com.name()));
        }
    }
}
long bruteMs = System.currentTimeMillis() - t0brute;

System.out.println("+- BRUTE FORCE --------------------------------------------------+");
System.out.printf( "|  %,d positions × %d communes = %,d comparisons%n",
    positions.size(), communes.size(),
    (long) positions.size() * communes.size());
System.out.printf( "|  Matches (incl. false positives) : %,d%n", bruteResults.size());
System.out.printf( "|  Time                            : %d ms%n", bruteMs);
System.out.println("+----------------------------------------------------------------+");
bruteResults.stream().limit(10)
    .forEach(m -> System.out.printf("  Veh %3d (Trip %3d) @ %-35s → %s%n",
        m.vehId(), m.tripId(), m.t(), m.commune()));

+- BRUTE FORCE --------------------------------------------------+
|  216,075 positions × 19 communes = 4,105,425 comparisons
|  Matches (incl. false positives) : 279,453
|  Time                            : 489 ms
+----------------------------------------------------------------+
  Veh   2 (Trip   2) @ 2020-06-01 08:17:54.798+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:17:56.298+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:17:57.048+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.248+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.256062+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.373654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:02.123654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:02.723654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:06.191997+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:07.

---
## <a id="n05"></a> 5 - Approach 2: RTree index (STBox first pass)

We index the 19 commune bounding boxes in an RTree, then for each vehicle position
we query the tree instead of scanning all communes.

> With only 19 communes the speedup is modest; the payoff becomes clear when
> the number of regions grows (hundreds of city blocks, thousands of parcels, etc.).

**How `rtree_search` returns results:**

`rtree_search` writes the count into a native integer (allocated with `Memory.allocate`)
and returns a native `int[]` of commune IDs (the indices we passed to `rtree_insert`).
We copy them into a Java array with `idsPtr.get(0, ids, 0, count)`.

In [10]:
// Build the index: one entry per commune (key = list index).
Pointer rtree = meos.rtree_create_stbox();
for (int i = 0; i < communes.size(); i++)
    meos.rtree_insert(rtree, communes.get(i).bbox(), i);
System.out.printf("RTree built: %d communes indexed%n%n", communes.size());

List<Match> rtreeResults = new ArrayList<>();
var rt = jnr.ffi.Runtime.getSystemRuntime();
Pointer countPtr = jnr.ffi.Memory.allocate(rt, Integer.BYTES);
long t0rtree = System.currentTimeMillis();

for (VehiclePos pos : positions) {
    countPtr.putInt(0, 0);
    Pointer idsPtr = meos.rtree_search(rtree, RTreeSearchOp.RTREE_OVERLAPS.getValue(), pos.bbox(), countPtr);
    int count = countPtr.getInt(0);
    //System.out.printf("DEBUG rtree_search → count = %d, idsPtr null = %b%n", scount, idsPtr == null);
    if (idsPtr != null && count > 0) {
        int[] ids = new int[count];
        idsPtr.get(0, ids, 0, count);
        for (int id : ids) {
            CommuneBox com = communes.get(id);
            rtreeResults.add(new Match(pos.vehId(), pos.tripId(), pos.t(), com.name()));
        }
    }
}
long rtreeMs = System.currentTimeMillis() - t0rtree;

System.out.println("+- RTREE INDEX --------------------------------------------------+");
System.out.printf( "|  Matches (incl. false positives) : %,d%n", rtreeResults.size());
System.out.printf( "|  Time                            : %d ms%n", rtreeMs);
System.out.println("+----------------------------------------------------------------+");
rtreeResults.stream().limit(10)
    .forEach(m -> System.out.printf("  Veh %3d (Trip %3d) @ %-35s → %s%n",
        m.vehId(), m.tripId(), m.t(), m.commune()));

RTree built: 19 communes indexed

+- RTREE INDEX --------------------------------------------------+
|  Matches (incl. false positives) : 279,453
|  Time                            : 481 ms
+----------------------------------------------------------------+
  Veh   2 (Trip   2) @ 2020-06-01 08:17:54.798+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:17:56.298+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:17:57.048+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.248+02          → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.256062+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:01.373654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:02.123654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:02.723654+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:06.191997+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:18:07.414288+02       → Uccle -

---
## <a id="n06"></a> 6 - Exact verification: eliminating false positives

The STBox of a commune is its **bounding rectangle**.
A vehicle position that overlaps the rectangle may still fall outside the actual commune
polygon (near a concave boundary, or in a corner shared with an adjacent commune):

```
   +----commune MBR-----+
   |  ╱╲                |
   | ╱  ╲   Y ← commune |      X = inside MBR but outside polygon
   |╱    ╲              |
   |  X   ╲____________ |
   +--------------------+
```

`geom_intersects2d(vehiclePoint, communePolygon)` returns `true` only if the point
lies inside (or on the boundary of) the actual polygon.

Both approaches (brute force and RTree) are timed independently with the exact check.

In [11]:
// Brute force + exact check
List<Match> bruteExact = new ArrayList<>();
long t0bruteExact = System.currentTimeMillis();
for (VehiclePos pos : positions) {
    for (CommuneBox com : communes) {
        if (meos.overlaps_stbox_stbox(pos.bbox(), com.bbox())
                && meos.geom_intersects2d(pos.geom(), com.geom())) {
            bruteExact.add(new Match(pos.vehId(), pos.tripId(), pos.t(), com.name()));
        }
    }
}
long bruteExactMs = System.currentTimeMillis() - t0bruteExact;

// RTree + exact check
List<Match> rtreeExact = new ArrayList<>();
var rt2 = jnr.ffi.Runtime.getSystemRuntime();
Pointer countPtr2 = jnr.ffi.Memory.allocate(rt2, Integer.BYTES);
long t0rtreeExact = System.currentTimeMillis();
for (VehiclePos pos : positions) {
    countPtr2.putInt(0, 0);
    Pointer idsPtr = meos.rtree_search(rtree, RTreeSearchOp.RTREE_OVERLAPS.getValue(), pos.bbox(), countPtr2);
    int count = countPtr2.getInt(0);
    if (idsPtr != null && count > 0) {
        int[] ids = new int[count];
        idsPtr.get(0, ids, 0, count);
        for (int id : ids) {
            CommuneBox com = communes.get(id);
            if (meos.geom_intersects2d(pos.geom(), com.geom())) {
                rtreeExact.add(new Match(pos.vehId(), pos.tripId(), pos.t(), com.name()));
            }
        }
    }
}
long rtreeExactMs = System.currentTimeMillis() - t0rtreeExact;

// Verify both approaches return the same results
var bruteSet = new TreeSet<>(bruteExact.stream()
    .map(m -> m.vehId() + "|" + m.tripId() + "|" + m.t() + "|" + m.commune())
    .collect(java.util.stream.Collectors.toList()));
var rtreeSet = new TreeSet<>(rtreeExact.stream()
    .map(m -> m.vehId() + "|" + m.tripId() + "|" + m.t() + "|" + m.commune())
    .collect(java.util.stream.Collectors.toList()));
boolean exactMatch = bruteSet.equals(rtreeSet);

// Sample of confirmed matches
System.out.println("Sample confirmed vehicle-commune assignments:");
System.out.println("-".repeat(75));
rtreeExact.stream().limit(15)
    .forEach(m -> System.out.printf("  Veh %3d (Trip %3d) @ %-35s → %s%n",
        m.vehId(), m.tripId(), m.t(), m.commune()));

// Per-vehicle commune visit
//
// We want to display, for each vehicle, how many instants it spent in each commune in order.
//
// Step 1 - build a nested map:  vehId → (commune → number of instants)
//
//   We use TreeMap for the outer map so vehicle IDs are automatically sorted
//   (TreeMap keeps its keys in ascending order, so vehicle 1 comes before 2, etc.).
//
//   Result structure:
//   {
//     2 → { "Anderlecht" → 1823, "Forest" → 412 },
//     5 → { "Ixelles"    → 934  },
//     ...
//   }
//
Map<Integer, Map<String, Long>> visitsByVehicle = new TreeMap<>();
for (Match m : rtreeExact) {
    // Get the inner map for this vehicle, creating it if it doesn't exist yet.
    Map<String, Long> communeCount =
        visitsByVehicle.computeIfAbsent(m.vehId(), id -> new HashMap<>());
    // Increment the counter for this commune (starting from 0 if first time seen).
    communeCount.put(m.commune(),
        communeCount.getOrDefault(m.commune(), 0L) + 1L);
}

// Step 2 - print results.
//
// visitsByVehicle.entrySet() returns all key-value pairs of the map as a Set.
// Each "entry" is one pair: entry.getKey() = vehicle id, entry.getValue() = commune map.
// Because the outer map is a TreeMap, entries are already in ascending key order
// (vehicle 1, then 2, then 3...) - no sorting needed here.
//
System.out.println();
System.out.println("Per-vehicle commune visit count (exact matches):");
for (Map.Entry<Integer, Map<String, Long>> vehicleEntry : visitsByVehicle.entrySet()) {
    int vehicleId              = vehicleEntry.getKey();   // e.g. 2
    Map<String, Long> communes = vehicleEntry.getValue(); // e.g. { "Anderlecht"→1823, ... }
    System.out.printf("  Vehicle %3d:%n", vehicleId);

    // Put commune entries in a list so we can sort them by visit count (descending).
    List<Map.Entry<String, Long>> communeList = new ArrayList<>(communes.entrySet());
    communeList.sort((a, b) -> Long.compare(b.getValue(), a.getValue())); // highest first

    for (Map.Entry<String, Long> ce : communeList) {
        String communeName = ce.getKey();   // e.g. "Anderlecht"
        long   instants    = ce.getValue(); // e.g. 1823
        System.out.printf("    %-40s %,5d instants%n", communeName, instants);
    }
}

Sample confirmed vehicle-commune assignments:
---------------------------------------------------------------------------
  Veh   2 (Trip   2) @ 2020-06-01 08:26:30.489763+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:30.82377+02        → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:31.46595+02        → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:32.7463+02         → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:33.4963+02         → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:33.987433+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:34.362433+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:34.722433+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:35.167106+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:36.653264+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:37.103064+02       → Uccle - Ukkel
  Veh   2 (Trip   2) @ 2020-06-01 08:26:37.603064+02 

---
## <a id="n07"></a> 7 - Summary table (19 communes)

Comparison of timing, match counts, and false positives for all four combinations
on the real 19 Brussels communes.

In [12]:
System.out.println("+------------------------------------------+----------+----------+");
System.out.println("|                                          |  Brute   |  RTree   |");
System.out.println("|                                          |  Force   |  Index   |");
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  STBox only (Sections 4 / 5)             | %5d ms | %4d ms  |%n",
    bruteMs, rtreeMs);
System.out.printf( "|  STBox + geom_intersects2d (Section 6)   | %5d ms | %4d ms  |%n",
    bruteExactMs, rtreeExactMs);
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  Matches - STBox only                    | %8d | %8d |%n",
    bruteResults.size(), rtreeResults.size());
System.out.printf( "|  Matches - exact check                   | %8d | %8d |%n",
    bruteExact.size(), rtreeExact.size());
System.out.printf( "|  False positives removed                 | %8d | %8d |%n",
    bruteResults.size() - bruteExact.size(),
    rtreeResults.size()  - rtreeExact.size());
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  RTree speedup - STBox only              |     %.1fx faster     |%n",
    (double) bruteMs / Math.max(rtreeMs, 1));
System.out.printf( "|  RTree speedup - exact check             |     %.1fx faster     |%n",
    (double) bruteExactMs / Math.max(rtreeExactMs, 1));
System.out.println("+------------------------------------------+----------+----------+");
System.out.println(exactMatch
    ? "|  ✓ Brute force and RTree return identical exact results        |"
    : "|  ✗ Results differ - check geometry construction               |");
System.out.println("+------------------------------------------+----------+----------+");

+------------------------------------------+----------+----------+
|                                          |  Brute   |  RTree   |
|                                          |  Force   |  Index   |
+------------------------------------------+----------+----------+
|  STBox only (Sections 4 / 5)             |   489 ms |  481 ms  |
|  STBox + geom_intersects2d (Section 6)   |  2564 ms | 2522 ms  |
+------------------------------------------+----------+----------+
|  Matches - STBox only                    |   279453 |   279453 |
|  Matches - exact check                   |   124571 |   124571 |
|  False positives removed                 |   154882 |   154882 |
+------------------------------------------+----------+----------+
|  RTree speedup - STBox only              |     1.0x faster     |
|  RTree speedup - exact check             |     1.0x faster     |
+------------------------------------------+----------+----------+
|  ✓ Brute force and RTree return identical exact results     

---
## <a id="n08"></a> 8 - Scalability benchmark: 19 / 100 / 500 regions

This section shows how all **four combinations** scale as the number of indexed regions grows:

| Approach | First pass | Second pass |
|---|---|---|
| Brute STBox | `overlaps_stbox_stbox` on all N | — |
| Brute Exact | `overlaps_stbox_stbox` on all N | `geom_intersects2d` on candidates |
| RTree STBox | `rtree_search` (log N) | — |
| RTree Exact | `rtree_search` (log N) | `geom_intersects2d` on candidates |

**Method:** the 19 real commune bounding boxes are tiled on a grid (50 km Y-shift per row).
For the exact check, each synthetic box is paired with its source commune polygon
(index `j % 19`). Since the polygon sits at the original position while the box is
shifted, `geom_intersects2d` correctly benchmarks the *cost of the elimination call*
without inflating the match count.

**Expected pattern:**
- At N = 19 the RTree overhead dominates → speedup ≈ 0.9×.
- At N = 100 the crossover point is approached.
- At N = 500 the RTree is clearly faster on both passes.

In [13]:
// Helper method - builds a shifted STBox from a real commune
//
// Parameters:
//   base   — index (0-18) of the commune to copy
//   shiftY — how many metres to move the box northward (Y axis in SRID 3857)
//
// Returns: a MEOS STBox Pointer with the same X extent as the real commune,
//          but shifted shiftY metres north.
//
Pointer makeShiftedBox(int base, double shiftY) {
    CommuneBox c = communes.get(base);
    String boxStr = String.format(java.util.Locale.US,
        "SRID=3857;STBOX X((%f %f),(%f %f))",
        c.xmin(), c.ymin() + shiftY,   // south-west corner (Y shifted north)
        c.xmax(), c.ymax() + shiftY);  // north-east corner (Y shifted north)
    return meos.stbox_in(boxStr);
}

// Scalability benchmark - 19 / 100 / 500 / 2000 indexed regions
//
// Problem: we only have 19 real communes but want to test with N = 100, 500...
//
// Solution - tile the 19 communes on a grid:
//   Copy the 19 boxes repeatedly, shifting each new row 50 km (arbitrary) northward.
//
//   Visual layout (each letter = one commune bounding box):
//
//   Row 2  (+100 km)   A B C D E F G H I J K L M N O P Q R S
//   Row 1   (+50 km)   A B C D E F G H I J K L M N O P Q R S
//   Row 0     (real)   A B C D E F G H I J K L M N O P Q R S  ← Brussels
//
// For box index j (from 0 to N-1):
//   base  = j % 19       → which of the 19 communes to copy
//                           (cycles: 0,1,...,18, 0,1,...,18, ...)
//   row   = j / 19       → which grid row (integer division: 0,0,...,0, 1,1,...,1, ...)
//   shift = row * 50 000 → Y offset in metres
//
// Concrete examples for N = 40:
//   j= 0 → commune  0, shift=      0 m  (row 0, real Anderlecht)
//   j=18 → commune 18, shift=      0 m  (row 0, last real commune)
//   j=19 → commune  0, shift= 50 000 m  (row 1, Anderlecht copied 50 km north)
//   j=37 → commune 18, shift= 50 000 m  (row 1, last commune of row 1)
//   j=38 → commune  0, shift=100 000 m  (row 2)
//   j=39 → commune  1, shift=100 000 m  (row 2)
//
final double OFFSET_M = 50_000.0; // 50 km between rows (SRID 3857, metres)
int[] scales = {19, 100, 500, 2000};

String hdr = "+-------+------------------+------------------+------------------+------------------+";
System.out.println(hdr);
System.out.println("|   N   | BruteSTBox  (ms) | BruteExact  (ms) | RTreeSTBox  (ms) | RTreeExact  (ms) |");
System.out.println(hdr);

for (int N : scales) {

    // Build the N synthetic bounding boxes
    List<Pointer> synBoxes = new ArrayList<>(N);
    // srcIdx[j] = index of the real commune whose polygon we reuse for exact check.
    int[] srcIdx = new int[N];
    for (int j = 0; j < N; j++) {
        int    base  = j % communes.size();             // which commune to copy
        double shift = (j / communes.size()) * OFFSET_M; // how far north (row * 50km)
        synBoxes.add(makeShiftedBox(base, shift));
        srcIdx[j] = base;
    }

    // 1. Brute force - STBox only
    // Loop over every vehicle × every box: O(positions × N).
    long t0 = System.currentTimeMillis();
    for (VehiclePos pos : positions)
        for (Pointer box : synBoxes)
            meos.overlaps_stbox_stbox(pos.bbox(), box);
    long msBruteStbox = System.currentTimeMillis() - t0;

    // 2. Brute force - STBox + exact check
    // Same loop, but for each STBox hit we also call geom_intersects2d.
    t0 = System.currentTimeMillis();
    for (VehiclePos pos : positions) {
        for (int j = 0; j < N; j++) {
            boolean stboxMatch = meos.overlaps_stbox_stbox(pos.bbox(), synBoxes.get(j));
            if (stboxMatch) {
                // The STBox overlaps — run the exact polygon test to eliminate
                // false positives. The polygon is at Brussels; the STBox is shifted
                // north, so this call almost always returns false, but we still pay
                // the full cost of the call — which is what we want to measure.
                meos.geom_intersects2d(pos.geom(), communes.get(srcIdx[j]).geom());
            }
        }
    }
    long msBruteExact = System.currentTimeMillis() - t0;

    // ── Build the RTree index for this scale ──────────────────────────────────
    // We insert all N boxes; the stored ID for each box is its index j.
    Pointer synTree = meos.rtree_create_stbox();
    for (int j = 0; j < N; j++)
        meos.rtree_insert(synTree, synBoxes.get(j), j);

    var rtSyn   = jnr.ffi.Runtime.getSystemRuntime();
    Pointer cpSyn = jnr.ffi.Memory.allocate(rtSyn, Integer.BYTES);

    // 3. RTree - STBox only
    // rtree_search returns only the candidate boxes (O(log N) traversal).
    t0 = System.currentTimeMillis();
    for (VehiclePos pos : positions) {
        cpSyn.putInt(0, 0);
        Pointer idsP = meos.rtree_search(synTree, RTreeSearchOp.RTREE_OVERLAPS.getValue(), pos.bbox(), cpSyn);
        int cnt = cpSyn.getInt(0);
        if (idsP != null && cnt > 0) {
            int[] ids = new int[cnt];
            idsP.get(0, ids, 0, cnt); // copy native int[] into Java array
        }
    }
    long msRtreeStbox = System.currentTimeMillis() - t0;

    // 4. RTree - STBox + exact check
    // Same as step 3, but we also run geom_intersects2d on each candidate.
    t0 = System.currentTimeMillis();
    for (VehiclePos pos : positions) {
        cpSyn.putInt(0, 0);
        Pointer idsP = meos.rtree_search(synTree, RTreeSearchOp.RTREE_OVERLAPS.getValue(), pos.bbox(), cpSyn);
        int cnt = cpSyn.getInt(0);
        if (idsP != null && cnt > 0) {
            int[] ids = new int[cnt];
            idsP.get(0, ids, 0, cnt);
            for (int id : ids) {
                // srcIdx[id] = the real commune polygon to test against.
                meos.geom_intersects2d(pos.geom(), communes.get(srcIdx[id]).geom());
            }
        }
    }
    long msRtreeExact = System.currentTimeMillis() - t0;
    meos.rtree_free(synTree);

    System.out.printf("| %5d | %16d | %16d | %16d | %16d |%n",
        N, msBruteStbox, msBruteExact, msRtreeStbox, msRtreeExact);
}

System.out.println(hdr);
System.out.printf("%nQuerying %,d vehicle positions per run.%n", positions.size());
System.out.println("Brute = O(pos × N).  RTree = O(pos × log N).");
System.out.println("Exact check cost visible in BruteExact vs BruteSTBox columns.");

+-------+------------------+------------------+------------------+------------------+
|   N   | BruteSTBox  (ms) | BruteExact  (ms) | RTreeSTBox  (ms) | RTreeExact  (ms) |
+-------+------------------+------------------+------------------+------------------+
|    19 |              181 |             2215 |              277 |             2239 |
|   100 |              862 |             2878 |              516 |             2710 |
|   500 |             4362 |             6147 |              552 |             2613 |
|  2000 |            18254 |            18914 |              611 |             2679 |
+-------+------------------+------------------+------------------+------------------+

Querying 216,075 vehicle positions per run.
Brute = O(pos × N).  RTree = O(pos × log N).
Exact check cost visible in BruteExact vs BruteSTBox columns.


---
## Appendix - Data files

| File | Format | CRS | Columns |
|---|---|---|---|
| `brussels_communes.csv` | CSV | SRID 3857 | `id, name, population, geom` (EWKB hex) |
| `berlinmod_instants.csv` | CSV | SRID 3857 | `tripid, vehid, day, seqno, geom, t` |

**Why SRID 3857 matters here:** both datasets share the same CRS, so coordinates
can be compared and intersected directly without any projection step. This is why
`geom_from_hexewkb` + `geo_to_stbox` works cleanly, unlike Exercise 4 where
the port geometries were in UTM 25832 and the AIS positions in WGS84.

**rapaio-jupyter-kernel reminders:**

| Rule | Detail |
|---|---|
| `%jars` must be alone in its cell | Mixing with Java or comments causes failure |
| JNR-FFI string returns crash the JVM | Declare as `Pointer`, read with `ptr.getString(0)` |
| `Locale.US` in `String.format` | Without it, commas in decimal numbers break WKT |
| `SRID=3857;STBOX X((x y),(x y))` | Point STBOX: both corners identical, SRID explicit |
| `stbox_xmin/ymin/xmax/ymax` | Use these instead of parsing `stbox_out` - locale-safe |
